In [1]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
import torch
import sys
from typing import List

In [2]:
module_path = "/home/ubuntu/Shree_FYP/train/stage2/training"

In [3]:
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
from updated_grpo_teacher import GRPOTeacher, RolloutBuffer

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [5]:
# ============================================================================
# Mock reward function for testing
# ============================================================================
class MockRewardFunction:
    """Simple reward: length-based + random noise"""
    def __call__(
        self,
        rollout_ids: torch.Tensor,
        rollout_text: List[str],
        pixel_values,
        image_grid_thw,
        ground_truth: dict,
    ) -> torch.Tensor:
        batch = rollout_ids.shape[0]
        # Reward = normalized length + small random component
        lengths = (rollout_ids != 0).sum(dim=-1).float()
        rewards = (lengths / lengths.max()) + 0.1 * torch.randn(batch, device=rollout_ids.device)
        return rewards

In [6]:
# ============================================================================
# Test 1: Instantiation
# ============================================================================
def test_instantiation():
    print("\n" + "="*70)
    print("TEST 1: Model Instantiation")
    print("="*70)
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            "/home/ubuntu/Shree_FYP/data/stage1_unsloth",
            trust_remote_code=True
        )
        
        # Add special tokens
        special_tokens = {"additional_special_tokens": ["<think>", "</think>", "<ans>", "</ans>"]}
        tokenizer.add_special_tokens(special_tokens)
        ans_id = tokenizer.convert_tokens_to_ids("<ans>")
        
        teacher = GRPOTeacher(
            pretrained_model_name_or_path="/home/ubuntu/Shree_FYP/data/stage1_unsloth",
            G=3,  # Use smaller G for testing
            answer_token_id=ans_id,
            lora_rank=16,  # Smaller for testing
            lora_alpha=32,
            gen_temperature=0.9,
            gen_max_new_tokens=64,  # Short for testing
            kl_coef=0.0,
        )
        teacher.vlm.to("cuda")
        
        print("✓ Teacher instantiated successfully")
        print(f"  - Answer token ID: {ans_id}")
        print(f"  - Hidden dim: {teacher.hidden_dim}")
        print(f"  - G (rollouts): {teacher.G}")
        
        teacher.print_trainable_parameters()
        
        return teacher, tokenizer
        
    except Exception as e:
        print(f"✗ Instantiation failed: {e}")
        raise

In [7]:
# ============================================================================
# Test 2: Forward Pass
# ============================================================================
def test_forward_pass(teacher, tokenizer):
    print("\n" + "="*70)
    print("TEST 2: Forward Pass")
    print("="*70)
    
    try:
        # Create dummy input
        prompt = "<think> Let me analyze this image. </think> <ans>"
        inputs = tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs.input_ids.to(teacher.vlm.device)
        attention_mask = inputs.attention_mask.to(teacher.vlm.device)
        
        # Forward pass
        with torch.no_grad():
            outputs = teacher.vlm(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                return_dict=True,
            )
        
        logits = outputs.logits
        print(f"✓ Forward pass successful")
        print(f"  - Input shape: {input_ids.shape}")
        print(f"  - Output logits shape: {logits.shape}")
        print(f"  - Logits dtype: {logits.dtype}")
        print(f"  - Logits range: [{logits.min():.2f}, {logits.max():.2f}]")
        
        return True
        
    except Exception as e:
        print(f"✗ Forward pass failed: {e}")
        raise

In [17]:
# ============================================================================
# Test 3: Generation
# ============================================================================
def test_generation(teacher, tokenizer):
    print("\n" + "="*70)
    print("TEST 3: Rollout Generation")
    print("="*70)
    
    try:
        # Create dummy batch
        prompts = [
            "<think> The robot should move to the red object. </think> <ans>",
            "<think> I need to grasp the blue cup carefully. </think> <ans>",
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True)
        input_ids = inputs.input_ids.to(teacher.vlm.device)
        attention_mask = inputs.attention_mask.to(teacher.vlm.device)
        
        # Generate rollouts
        all_ids, all_texts, all_masks = teacher.generate_rollouts(
            input_ids=input_ids,
            pixel_values=None,
            image_grid_thw=None,
            attention_mask=attention_mask,
            tokenizer=tokenizer,
        )
        
        print(f"✓ Generation successful")
        print(f"  - Generated {len(all_ids)} rollouts (G={teacher.G})")
        print(f"  - Batch size: {input_ids.shape[0]}")
        print(f"  - Sequence lengths: {[ids.shape[1] for ids in all_ids]}")
        
        # Show sample generation
        print(f"\n  Sample generation (rollout 0, batch item 0):")
        print(f"  {all_texts[0][0]}...")
        
        return all_ids, all_texts, all_masks, input_ids.shape[1]
        
    except Exception as e:
        print(f"✗ Generation failed: {e}")
        raise

In [18]:
# ============================================================================
# Test 4: Reward Scoring
# ============================================================================
def test_reward_scoring(teacher, all_ids, all_texts):
    print("\n" + "="*70)
    print("TEST 4: Reward Scoring")
    print("="*70)
    
    try:
        reward_fn = MockRewardFunction()
        
        rewards = teacher.score_rollouts(
            all_ids=all_ids,
            all_texts=all_texts,
            pixel_values=None,
            image_grid_thw=None,
            ground_truth={},
            reward_fns=[reward_fn],
        )
        
        print(f"✓ Reward scoring successful")
        print(f"  - Rewards shape: {rewards.shape} (expected: [{teacher.G}, batch])")
        print(f"  - Rewards per rollout:")
        for g in range(teacher.G):
            print(f"    Rollout {g}: {rewards[g].tolist()}")
        
        return rewards
        
    except Exception as e:
        print(f"✗ Reward scoring failed: {e}")
        raise

In [19]:
# ============================================================================
# Test 5: Advantage Computation
# ============================================================================
def test_advantage_computation(teacher, rewards):
    print("\n" + "="*70)
    print("TEST 5: Advantage Computation")
    print("="*70)
    
    try:
        advantages = teacher.compute_advantages(rewards)
        
        print(f"✓ Advantage computation successful")
        print(f"  - Advantages shape: {advantages.shape}")
        print(f"  - Mean per batch item: {advantages.mean(dim=0).tolist()}")
        print(f"  - Std per batch item: {advantages.std(dim=0).tolist()}")
        
        # Check normalization
        mean_check = advantages.mean(dim=0).abs().max().item()
        std_check = advantages.std(dim=0).mean().item()
        
        print(f"  - Mean ≈ 0 check: {mean_check:.6f} (should be ~0)")
        print(f"  - Std ≈ 1 check: {std_check:.6f} (should be ~1)")
        
        if mean_check < 1e-5 and 0.9 < std_check < 1.1:
            print("  ✓ Advantages properly normalized")
        else:
            print("  ⚠ Advantages may not be properly normalized")
        
        return advantages
        
    except Exception as e:
        print(f"✗ Advantage computation failed: {e}")
        raise

In [20]:
# ============================================================================
# Test 6: GRPO Loss Computation
# ============================================================================
def test_grpo_loss(teacher, all_ids, all_masks, advantages, prompt_len):
    print("\n" + "="*70)
    print("TEST 6: GRPO Loss Computation")
    print("="*70)
    
    try:
        # Enable gradients
        teacher.vlm.train()
        
        loss = teacher.compute_grpo_loss(
            all_ids=all_ids,
            all_masks=all_masks,
            advantages=advantages,
            pixel_values=None,
            image_grid_thw=None,
            prompt_len=prompt_len,
        )
        
        print(f"✓ GRPO loss computation successful")
        print(f"  - Loss value: {loss.item():.6f}")
        print(f"  - Loss dtype: {loss.dtype}")
        print(f"  - Requires grad: {loss.requires_grad}")
        
        # Test backward pass
        loss.backward()
        
        # Check gradients exist
        grad_norm = 0.0
        for p in teacher.vlm.parameters():
            if p.grad is not None:
                grad_norm += p.grad.norm().item() ** 2
        grad_norm = grad_norm ** 0.5
        
        print(f"  - Backward pass successful")
        print(f"  - Gradient norm: {grad_norm:.6f}")
        
        if grad_norm > 0:
            print("  ✓ Gradients flowing correctly")
        else:
            print("  ⚠ No gradients detected - check LoRA configuration")
        
        # Zero gradients for next test
        teacher.vlm.zero_grad()
        
        return loss
        
    except Exception as e:
        print(f"✗ GRPO loss computation failed: {e}")
        raise

In [21]:
# ============================================================================
# Test 7: Best/Worst Selection
# ============================================================================
def test_best_worst_selection(teacher, all_ids, all_masks, all_texts, advantages, prompt_len):
    print("\n" + "="*70)
    print("TEST 7: Best/Worst Selection (τ+/τ-)")
    print("="*70)
    
    try:
        result = teacher.select_best_worst(
            all_ids=all_ids,
            all_masks=all_masks,
            all_texts=all_texts,
            advantages=advantages,
            prompt_len=prompt_len,
        )
        
        (tau_pos_ids, tau_pos_mask, tau_neg_ids, tau_neg_mask,
         tau_pos_texts, tau_neg_texts, tau_pos_response, tau_neg_response,
         answer_pos) = result
        
        print(f"✓ Best/worst selection successful")
        print(f"  - τ+ shape: {tau_pos_ids.shape}")
        print(f"  - τ- shape: {tau_neg_ids.shape}")
        print(f"  - Answer positions: {answer_pos.tolist()}")
        
        # Show selected advantages
        best_idx = advantages.argmax(dim=0)
        worst_idx = advantages.argmin(dim=0)
        
        print(f"  - Best rollout indices: {best_idx.tolist()}")
        print(f"  - Worst rollout indices: {worst_idx.tolist()}")
        
        for i in range(len(best_idx)):
            print(f"    Batch {i}: best_adv={advantages[best_idx[i], i]:.3f}, "
                  f"worst_adv={advantages[worst_idx[i], i]:.3f}")
        
        return result
        
    except Exception as e:
        print(f"✗ Best/worst selection failed: {e}")
        raise

In [22]:
# ============================================================================
# Test 8: Hidden State Extraction
# ============================================================================
def test_hidden_state_extraction(teacher, tau_pos_ids, tau_pos_mask, answer_pos):
    print("\n" + "="*70)
    print("TEST 8: Hidden State Extraction (h_T)")
    print("="*70)
    
    try:
        h_T = teacher.extract_answer_hidden_state(
            tau_pos_ids=tau_pos_ids,
            tau_pos_mask=tau_pos_mask,
            pixel_values=None,
            image_grid_thw=None,
            answer_token_pos=answer_pos,
        )
        
        print(f"✓ Hidden state extraction successful")
        print(f"  - h_T shape: {h_T.shape} (expected: [batch, {teacher.hidden_dim}])")
        print(f"  - h_T dtype: {h_T.dtype}")
        print(f"  - h_T norm per sample: {h_T.norm(dim=-1).tolist()}")
        
        # Check that h_T is not all zeros
        if h_T.abs().sum() > 0:
            print("  ✓ Hidden states are non-zero")
        else:
            print("  ⚠ Hidden states are all zeros - check answer_token_id")
        
        return h_T
        
    except Exception as e:
        print(f"✗ Hidden state extraction failed: {e}")
        raise

In [23]:
# ============================================================================
# Test 9: Full Training Step
# ============================================================================
def test_full_training_step(teacher, tokenizer):
    print("\n" + "="*70)
    print("TEST 9: Full Training Step (End-to-End)")
    print("="*70)
    
    try:
        # Create optimizer
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, teacher.vlm.parameters()),
            lr=1e-4,
        )
        
        # Create dummy batch
        prompts = [
            "<think> Analyze the scene and plan the robot movement. </think> <ans>",
            "<think> The object is on the table, need to grasp it. </think> <ans>",
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True)
        input_ids = inputs.input_ids.to(teacher.vlm.device)
        attention_mask = inputs.attention_mask.to(teacher.vlm.device)
        
        reward_fn = MockRewardFunction()
        
        # Run full training step
        buffer = teacher.training_step(
            input_ids=input_ids,
            pixel_values=None,
            image_grid_thw=None,
            attention_mask=attention_mask,
            ground_truth={},
            reward_fns=[reward_fn],
            reward_weights=None,
            optimizer=optimizer,
            tokenizer=tokenizer,
            grad_clip=1.0,
        )
        
        print(f"✓ Full training step successful")
        print(f"  - RolloutBuffer created: {type(buffer).__name__}")
        print(f"  - Rewards: {buffer.rewards.shape}")
        print(f"  - Advantages: {buffer.advantages.shape}")
        print(f"  - h_T: {buffer.h_T.shape}")
        print(f"  - τ+ ids: {buffer.tau_pos_ids.shape}")
        print(f"  - τ- ids: {buffer.tau_neg_ids.shape}")
        
        # Log stats
        stats = teacher.log_rollout_stats(buffer)
        print(f"\n  Rollout statistics:")
        for k, v in stats.items():
            print(f"    {k}: {v:.4f}")
        
        return buffer
        
    except Exception as e:
        print(f"✗ Full training step failed: {e}")
        raise

In [24]:
# ============================================================================
# Main test runner
# ============================================================================
def main():
    print("\n" + "="*70)
    print("GRPO TEACHER COMPREHENSIVE TEST SUITE")
    print("="*70)
    
    # Test 1: Instantiation
    teacher, tokenizer = test_instantiation()
    
    # Test 2: Forward pass
    test_forward_pass(teacher, tokenizer)
    
    # Test 3: Generation
    all_ids, all_texts, all_masks, prompt_len = test_generation(teacher, tokenizer)
    
    # Test 4: Reward scoring
    rewards = test_reward_scoring(teacher, all_ids, all_texts)
    
    # Test 5: Advantage computation
    advantages = test_advantage_computation(teacher, rewards)
    
    # Test 6: GRPO loss
    test_grpo_loss(teacher, all_ids, all_masks, advantages, prompt_len)
    
    # Test 7: Best/worst selection
    result = test_best_worst_selection(
        teacher, all_ids, all_masks, all_texts, advantages, prompt_len
    )
    tau_pos_ids, tau_pos_mask = result[0], result[1]
    answer_pos = result[8]
    
    # Test 8: Hidden state extraction
    test_hidden_state_extraction(teacher, tau_pos_ids, tau_pos_mask, answer_pos)
    
    # Test 9: Full training step
    test_full_training_step(teacher, tokenizer)
    
    print("\n" + "="*70)
    print("ALL TESTS PASSED ✓")
    print("="*70)
    print("\nYour Teacher is ready for training!")
    print("You can now integrate it into your main training loop.")

In [25]:
main()


GRPO TEACHER COMPREHENSIVE TEST SUITE

TEST 1: Model Instantiation


The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

✓ Teacher instantiated successfully
  - Answer token ID: 248077
  - Hidden dim: 2560
  - G (rollouts): 3
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101

TEST 2: Forward Pass
✓ Forward pass successful
  - Input shape: torch.Size([1, 11])
  - Output logits shape: torch.Size([1, 11, 248320])
  - Logits dtype: torch.bfloat16
  - Logits range: [-12.69, 20.50]

TEST 3: Rollout Generation
✓ Generation successful
  - Generated 3 rollouts (G=3)
  - Batch size: 2
  - Sequence lengths: [78, 63, 78]

  Sample generation (rollout 0, batch item 0):
   Fine-Tuned Large Language Model and Robot

The task involves finding and moving to a red object. I'll analyze the image and describe my next move.

Based on the image, I can see a red object on the table. The robot should:

1. Move in the direction of the table
2....

TEST 4: Reward Scoring
✓ Reward scoring successful
  - Rewards shape: torch.Size([3, 2]) (expected: [3, batch])
  - Rewards per rollout:
    Rollout 0: [1